# TMDWF N-State Fit Template

This notebook is a template wrapper around the repository's TMDWF ratio-fit workflow.
Edit the input block below, validate it, and then run the same backend used by the CLI.


## Imports / Setup

Run this notebook from the repository root, or adjust `REPO_ROOT` below.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from lqcd_analysis.notebook_workflows import (
    pretty_print_config,
    render_tmdwf_fit_input_text,
    run_tmdwf_fit_from_notebook,
    validate_tmdwf_notebook_config,
)


## User Inputs

These fields mirror the plain-text TMDWF input file format.
This template is structurally complete, but you should point it at your own HDF5 data and two-point plateau tables.


In [ ]:
EXAMPLE_C2PT = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_c2pt_csv"
EXAMPLE_qTMDWF = REPO_ROOT / "examples" / "data" / "l64c64a076_m140" / "comb_qtmdwf"
EXAMPLE_OUTPUTS = REPO_ROOT / "examples" / "outputs" / "tmdwf_fit_notebook"

workflow_config = {
    # Data settings
    "title_pattern": "l64c64a076_m140_tmdwf_pz*",
    "ns": 64,
    "nt": 64,
    "lattice_spacing_fm": 0.076,
    "pzlist": [0],
    "gmlist": ["T5"],  # use ["Z5"] for gamma_z gamma_5
    "etalist": ["eta0"],
    "Tdirlist": ["plus", "minus"],
    "bTlist": [0],
    "bzlist": [0],
    "qtmdwf_h5": str(EXAMPLE_qTMDWF / "qtmdwf_pz*.h5"),
    "dataset_path_template": "{gm}/{eta}/pz{pz}/{Tdir}/bT{bT}/bz{bz}",
    "two_point_plateau_table": "/path/to/2pt_plateau_pz*_tmax#_plateau.txt",
    "c2pt": str(EXAMPLE_C2PT / "c2pt_5_5_k0_pz*_real.csv"),
    "fold_t": "periodic",
    "tsrange": [0, 20],

    # Fit settings
    "fit_target": "ratio",
    "fit_component": "both",
    "nstates": [1, 2],
    "binsize": 1,
    "bootstrap_samples": 32,
    "bootstrap_size": 32,
    "seed": 2026,
    "tmin": 2,
    "tmax": "auto",
    "shared_window_by_pz_gm": False,
    # Used only when shared_window_by_pz_gm is True
    # "decay_constant": [0.11, 0.03],
    # "min_fit_dof": 1,
    "plot": False,

    # Output settings
    "results_dir": str(EXAMPLE_OUTPUTS),
}


## Option Guide

Edit only `workflow_config` in the cell above for normal usage.
The keys are grouped by comments so data settings, fit settings, and output settings stay easy to scan.

- `title_pattern`: Output title pattern. Use `*` where the momentum index `pz` should be inserted.
- `ns`, `nt`: Spatial and temporal lattice extents.
- `lattice_spacing_fm`: Stored in metadata and summaries.
- `fit_target`: Keep this as `"ratio"` in the first implementation.
- `fit_component`: Choose `"real"`, `"imag"`, or `"both"`.
- `nstates`: Supported values are `1`, `2`, or `[1, 2]`.
- `pzlist`: Integer momentum labels to analyze.
- `gmlist`, `etalist`, `Tdirlist`: Lists passed into the HDF5 dataset-path expansion. Supported labels in the first version are `"T5"` for gamma_t gamma_5 and `"Z5"` for gamma_z gamma_5.
- `bTlist` / `bTrange`: Transverse-separation choices. Provide one style only.
- `bzlist` / `bzrange`: Longitudinal-separation choices. When `bz != 0`, the backend combines `+bz` and `-bz` automatically.
- `qtmdwf_h5`: HDF5 file path or wildcard pattern.
- `dataset_path_template`: HDF5 dataset template with placeholders `{gm}`, `{eta}`, `{pz}`, `{Tdir}`, `{bT}`, and `{bz}`.
- `two_point_plateau_table`: Two-point plateau table that supplies fixed `A_i` and `E_i` central values. You may use `*` for the `pz` wildcard and `#` as a numeric wildcard for `_tmax<digits>_plateau.txt` matching. A practical example is `/path/to/2pt_plateau_pz*_tmax#_plateau.txt`. When `#` matches multiple files, the backend selects the largest inferred `tmax`.
- `c2pt`: Two-point correlator CSV used for the denominator of the ratio.
- `fold_t`: Folding mode for the denominator correlator. Use the same convention as the matching two-point analysis.
- Operator behavior: `T5` keeps the original sign-pattern preprocessing and numerator model. `Z5` multiplies the correlator by `-i` before folding and uses the extra lattice-momentum factor `Pz/E_i` in the numerator model.
- `tsrange`: Optional raw time range kept before fitting. If omitted, the backend defaults to `[0, Nt//2 - 1]`.
- `binsize`, `bootstrap_samples`, `bootstrap_size`, `seed`: Bootstrap controls.
- `tmin`: Starting time for the fit window. When `shared_window_by_pz_gm` is enabled, the backend scans upward from this value on the reference dataset.
- `tmax`: Optional. An explicit integer always wins. If set to `"auto"` or omitted, the backend infers `tmax` from the selected two-point plateau filename token `_tmax<digits>_plateau.txt`.
- `shared_window_by_pz_gm`: Optional boolean. When `true`, choose one common `(tmin, tmax)` per `(pz, gm, nstates)` from the reference dataset `(eta=etalist[0], bT=0, bz=0)` and reuse it for all `eta / bT / bz` combinations under that `(pz, gm, nstates)`.
- `decay_constant`: Required when `shared_window_by_pz_gm` is `true`. Provide `[value, error]`. The shared window is ranked by how the fitted `m0` plateau window overlaps this target.
- `min_fit_dof`: Optional minimum fit degrees of freedom for shared-window candidate rows. Default is `1`.
- Output grouping: for a fixed `(title, gm, eta, bT, component, nstates)`, the workflow writes one grouped summary / fit / samples / curve file containing all `bz` entries.
- Grouped summaries contain one parseable `begin_bz ...` / `end_bz ...` block per `bz`, along with `two_point_plateau_table_resolved`, `two_point_tmax_source`, and `two_point_tmax_inferred`.
- Grouped fit tables include metadata columns such as `shared_window_flag`, `reference_eta`, `reference_bT`, `reference_bz`, and `plateau_tmax_used`.
Lower fit bound for the TMDWF ratio fit.
- `tmax`: Optional upper fit bound. An explicit integer always wins. If set to `"auto"` or omitted, the backend infers it from the `two_point_plateau_table` filename token `_tmax<digits>_plateau.txt`.
- `plot`: Reserved for later extension; keep `False` in this first version.
- `results_dir`: Output directory. If set to `None`, notebook runs default to the notebook directory.


## Validate Config


In [ ]:
parsed = validate_tmdwf_notebook_config(workflow_config)
parsed


## Render Plain-Text Input Preview


In [ ]:
print(render_tmdwf_fit_input_text(workflow_config))


## Run Backend Workflow


In [ ]:
outputs = run_tmdwf_fit_from_notebook(workflow_config)
for output in outputs:
    print(output)


## Config Snapshot


In [ ]:
print(pretty_print_config(workflow_config))
